# 03 - Fusion Dataset Builder

This notebook builds a training-ready multimodal fusion dataset by:

1. Loading preprocessed window-level physiological features
2. Loading optional modality tables (EEG, speech, AUs, NLP) when available
3. Loading labels (if available) and creating fallback pseudo-labels
4. Harmonizing keys and joining tables safely
5. Generating modality-presence flags and completeness diagnostics
6. Exporting fusion-ready CSV files and metadata manifests

In [1]:
# SECTION 1: Imports

import json
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')

print('✓ Imports loaded')

✓ Imports loaded


In [2]:
# SECTION 2: Configuration and Paths

@dataclass
class FusionBuildConfig:
    use_normalized_physio: bool = True
    require_labels: bool = False
    create_pseudo_labels_if_missing: bool = True
    pseudo_label_strategy: str = 'task_rule'  # task_rule | none
    min_modalities_required: int = 1
    expect_multimodal: bool = True
    strict_multimodal_check: bool = False
    output_version: str = 'v1_fusion'


CONFIG = FusionBuildConfig()

# Detect environment
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    mounted = False
    try:
        drive.mount('/content/drive', force_remount=False)
        mounted = True
    except Exception as e:
        print(f'⚠️ Drive mount failed, using /content fallback: {e}')

    if mounted:
        DATASET_ROOT = Path('/content/drive/MyDrive/MultiPhysio-HRC')
        PREPROC_ROOT = Path('/content/drive/MyDrive/research_outputs/preprocessing/v1_windowed')
        OUTPUT_ROOT = Path('/content/drive/MyDrive/research_outputs/fusion')
    else:
        DATASET_ROOT = Path('/content/MultiPhysio-HRC')
        PREPROC_ROOT = Path('/content/research_outputs/preprocessing/v1_windowed')
        OUTPUT_ROOT = Path('/content/research_outputs/fusion')
else:
    DATASET_ROOT = Path.home() / 'Desktop' / 'thesis' / 'data' / 'multiphysio_hrc'
    PREPROC_ROOT = Path.home() / 'Desktop' / 'thesis' / 'phd_project' / 'sensor_fusion' / 'outputs' / 'preprocessing' / 'v1_windowed'
    OUTPUT_ROOT = Path.home() / 'Desktop' / 'thesis' / 'phd_project' / 'sensor_fusion' / 'outputs' / 'fusion'

FEATURES_ROOT = DATASET_ROOT / 'features'
OUT_DIR = OUTPUT_ROOT / CONFIG.output_version
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('✓ Paths configured')
print(f'  DATASET_ROOT: {DATASET_ROOT}')
print(f'  FEATURES_ROOT exists: {FEATURES_ROOT.exists()}')
print(f'  PREPROC_ROOT: {PREPROC_ROOT} (exists: {PREPROC_ROOT.exists()})')
print(f'  OUT_DIR: {OUT_DIR}')
print('  CONFIG:', asdict(CONFIG))

Mounted at /content/drive
✓ Paths configured
  DATASET_ROOT: /content/drive/MyDrive/MultiPhysio-HRC
  FEATURES_ROOT exists: True
  PREPROC_ROOT: /content/drive/MyDrive/research_outputs/preprocessing/v1_windowed (exists: True)
  OUT_DIR: /content/drive/MyDrive/research_outputs/fusion/v1_fusion
  CONFIG: {'use_normalized_physio': True, 'require_labels': False, 'create_pseudo_labels_if_missing': True, 'pseudo_label_strategy': 'task_rule', 'min_modalities_required': 1, 'expect_multimodal': True, 'strict_multimodal_check': False, 'output_version': 'v1_fusion'}


In [3]:
# SECTION 3: Load Core Preprocessed Physiological Windows

def load_physio_windows(preproc_root: Path, use_normalized: bool = True) -> pd.DataFrame:
    preferred = 'window_features_normalized.csv' if use_normalized else 'window_features_raw.csv'
    fallback = 'window_features_raw.csv' if use_normalized else 'window_features_normalized.csv'

    p1 = preproc_root / preferred
    p2 = preproc_root / fallback

    if p1.exists():
        path = p1
    elif p2.exists():
        path = p2
        print(f'⚠️ Preferred file missing, using fallback: {path.name}')
    else:
        raise FileNotFoundError('No window feature CSV found in preprocessing output folder.')

    df = pd.read_csv(path)
    return df


physio_df = load_physio_windows(PREPROC_ROOT, use_normalized=CONFIG.use_normalized_physio)

# Core keys expected from preprocessing notebook
core_keys = ['subject_id', 'task_name', 'task_file', 'split', 'window_idx', 'start_idx', 'end_idx', 'n_samples']
missing_core = [c for c in core_keys if c not in physio_df.columns]
if missing_core:
    raise ValueError(f'Missing core key columns in physio table: {missing_core}')

print('✓ Physiological window table loaded')
print(f'  Shape: {physio_df.shape}')
print(f'  Subjects: {physio_df.subject_id.nunique()}')
print(f'  Tasks: {physio_df.task_name.nunique()}')
print(f'  Windows: {len(physio_df)}')

display(physio_df.head(3))

✓ Physiological window table loaded
  Shape: (5640, 152)
  Subjects: 52
  Tasks: 21
  Windows: 5640


,subject_id,task_name,task_file,split,ID__mean,ID__std,ID__min,ID__max,ID__median,ID__p25,ID__p75,ID__energy,Repetition__mean,Repetition__std,Repetition__min,Repetition__max,Repetition__median,Repetition__p25,Repetition__p75,Repetition__energy,ECG__mean,ECG__std,ECG__min,ECG__max,ECG__median,ECG__p25,ECG__p75,ECG__energy,EDA__mean,EDA__std,EDA__min,EDA__max,EDA__median,EDA__p25,EDA__p75,EDA__energy,EMG__mean,EMG__std,EMG__min,EMG__max,EMG__median,EMG__p25,EMG__p75,EMG__energy,RESP__mean,RESP__std,RESP__min,RESP__max,RESP__median,RESP__p25,RESP__p75,RESP__energy,EEG_channel_0__mean,EEG_channel_0__std,EEG_channel_0__min,EEG_channel_0__max,EEG_channel_0__median,EEG_channel_0__p25,EEG_channel_0__p75,EEG_channel_0__energy,EEG_channel_1__mean,EEG_channel_1__std,EEG_channel_1__min,EEG_channel_1__max,EEG_channel_1__median,EEG_channel_1__p25,EEG_channel_1__p75,EEG_channel_1__energy,EEG_channel_2__mean,EEG_channel_2__std,EEG_channel_2__min,EEG_channel_2__max,EEG_channel_2__median,EEG_channel_2__p25,EEG_channel_2__p75,EEG_channel_2__energy,EEG_channel_3__mean,EEG_channel_3__std,EEG_channel_3__min,EEG_channel_3__max,EEG_channel_3__median,EEG_channel_3__p25,EEG_channel_3__p75,EEG_channel_3__energy,EEG_channel_4__mean,EEG_channel_4__std,EEG_channel_4__min,EEG_channel_4__max,EEG_channel_4__median,EEG_channel_4__p25,EEG_channel_4__p75,EEG_channel_4__energy,EEG_channel_5__mean,EEG_channel_5__std,EEG_channel_5__min,EEG_channel_5__max,EEG_channel_5__median,EEG_channel_5__p25,EEG_channel_5__p75,EEG_channel_5__energy,EEG_channel_6__mean,EEG_channel_6__std,EEG_channel_6__min,EEG_channel_6__max,EEG_channel_6__median,EEG_channel_6__p25,EEG_channel_6__p75,EEG_channel_6__energy,EEG_channel_7__mean,EEG_channel_7__std,EEG_channel_7__min,EEG_channel_7__max,EEG_channel_7__median,EEG_channel_7__p25,EEG_channel_7__p75,EEG_channel_7__energy,EEG_channel_8__mean,EEG_channel_8__std,EEG_channel_8__min,EEG_channel_8__max,EEG_channel_8__median,EEG_channel_8__p25,EEG_channel_8__p75,EEG_channel_8__energy,EEG_channel_9__mean,EEG_channel_9__std,EEG_channel_9__min,EEG_channel_9__max,EEG_channel_9__median,EEG_channel_9__p25,EEG_channel_9__p75,EEG_channel_9__energy,EEG_channel_10__mean,EEG_channel_10__std,EEG_channel_10__min,EEG_channel_10__max,EEG_channel_10__median,EEG_channel_10__p25,EEG_channel_10__p75,EEG_channel_10__energy,EEG_channel_11__mean,EEG_channel_11__std,EEG_channel_11__min,EEG_channel_11__max,EEG_channel_11__median,EEG_channel_11__p25,EEG_channel_11__p75,EEG_channel_11__energy,window_idx,start_idx,end_idx,n_samples
0,10000,cobot-task-1,cobot-task-1.csv,filtered,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.048811,0.002840,0.985618,0.002348,0.335566,0.981420,0.001114,0.000008,0.037181,0.001359,0.045884,0.030418,0.038100,0.041787,0.033337,0.962817,0.524582,0.507086,0.807671,0.054055,0.640794,0.48483,0.518462,0.266931,0.542639,0.000148,0.999937,0.000521,0.232745,0.992245,0.000340,2.883220e-08,0.947833,0.002661,0.995993,0.007329,0.658837,0.916780,0.007530,0.000012,0.492878,0.002321,0.991385,0.001082,0.638098,0.991739,0.014541,0.000009,0.498772,0.001915,0.996774,0.005957,0.689914,0.998286,0.013933,0.000006,0.498756,0.001023,0.998567,0.001288,0.732596,0.997671,0.018906,0.000003,0.942902,0.002598,0.994240,0.007731,0.597533,0.975079,0.002461,0.000010,0.922905,0.005018,0.992225,0.025194,0.455145,0.969907,0.006241,0.000032,0.752640,0.000489,0.998587,0.002793,0.852715,0.999233,0.000852,6.649730e-07,0.847278,0.015472,0.976751,0.075753,0.787997,0.994808,0.001150,0.000255,0.792107,0.001149,0.998044,0.004981,0.862076,0.997817,0.000770,2.554407e-06,0.697146,0.002591,0.991501,0.002298,0.320937,0.998670,0.001185,0.000010,0.841469,0.000543,0.999154,0.001311,0.761107,0.998467,0.000560,9.261405e-07,0.769014,0.001415,0.996929,0.004799,0.713396,0.998855,0.000879,3.372346e-06,0,0,15360,15360
1,10000,cobot-task-1,cobot-task-1.csv,filtered,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.099927,0.002990,0.999256,0.002066,0.320363,0.980634,0.0002

In [4]:
# SECTION 4: Optional Modality Table Loaders (Auto-discovery)


def _safe_read_csv(path: Path) -> Optional[pd.DataFrame]:
    if path.exists():
        try:
            return pd.read_csv(path)
        except Exception as e:
            print(f'⚠️ Failed to read {path}: {e}')
            return None
    return None


def _dedupe_paths(paths: List[Path]) -> List[Path]:
    seen = set()
    out = []
    for p in paths:
        s = str(p.resolve()) if p.exists() else str(p)
        if s not in seen:
            seen.add(s)
            out.append(p)
    return out


def discover_modality_file(search_roots: List[Path], exact_names: List[str], glob_patterns: List[str]) -> Optional[Path]:
    # 1) Exact name search first
    exact_candidates = []
    for root in search_roots:
        if not root.exists():
            continue
        for name in exact_names:
            p = root / name
            if p.exists() and p.is_file():
                exact_candidates.append(p)
    exact_candidates = sorted(_dedupe_paths(exact_candidates), key=lambda x: len(str(x)))
    if exact_candidates:
        return exact_candidates[0]

    # 2) Recursive glob fallback
    glob_candidates = []
    for root in search_roots:
        if not root.exists():
            continue
        for pat in glob_patterns:
            try:
                glob_candidates.extend([p for p in root.rglob(pat) if p.is_file()])
            except Exception:
                pass

    glob_candidates = sorted(_dedupe_paths(glob_candidates), key=lambda x: len(str(x)))
    if glob_candidates:
        return glob_candidates[0]

    return None


SEARCH_ROOTS = [
    FEATURES_ROOT,
    DATASET_ROOT / 'features',
    DATASET_ROOT,
]

FILE_HINTS = {
    'labels': {
        'exact': ['labels.csv'],
        'glob': ['*label*.csv', '*labels*.csv'],
    },
    'eeg_features_5s': {
        'exact': ['eeg_features_5s.csv'],
        'glob': ['*eeg*feature*.csv', '*eeg*.csv'],
    },
    'speech_features': {
        'exact': ['speech_features.csv'],
        'glob': ['*speech*feature*.csv', '*audio*feature*.csv', '*speech*.csv'],
    },
    'aus_data': {
        'exact': ['aus_data.csv'],
        'glob': ['*au*.csv', '*action*unit*.csv', '*facial*.csv'],
    },
    'nlp_embeddings': {
        'exact': ['nlp_embeddings.csv'],
        'glob': ['*nlp*embed*.csv', '*text*embed*.csv', '*embedding*.csv'],
    },
}

modality_sources = {}
modality_tables = {}

for mod, hints in FILE_HINTS.items():
    p = discover_modality_file(SEARCH_ROOTS, hints['exact'], hints['glob'])
    modality_sources[mod] = p
    modality_tables[mod] = _safe_read_csv(p) if p is not None else None

print('✓ Optional modality tables scan complete (auto-discovery)')
print('  Search roots:')
for root in SEARCH_ROOTS:
    print(f'    - {root} (exists: {root.exists()})')

for k, v in modality_tables.items():
    src = modality_sources.get(k)
    src_text = str(src) if src is not None else 'not found'
    status = f'shape={v.shape}' if v is not None else 'missing'
    print(f'  {k}: {status} | source: {src_text}')

✓ Optional modality tables scan complete (auto-discovery)
  Search roots:
    - /content/drive/MyDrive/MultiPhysio-HRC/features (exists: True)
    - /content/drive/MyDrive/MultiPhysio-HRC/features (exists: True)
    - /content/drive/MyDrive/MultiPhysio-HRC (exists: True)
  labels: missing | source: not found
  eeg_features_5s: shape=(5640, 104) | source: /content/drive/MyDrive/MultiPhysio-HRC/features/eeg_features_5s.csv
  speech_features: missing | source: not found
  aus_data: missing | source: not found
  nlp_embeddings: missing | source: not found


In [5]:
# SECTION 5: Label Preparation (Real Labels or Pseudo-Labels)


def create_task_rule_label(task_name: str) -> str:
    name = str(task_name).lower()
    if any(x in name for x in ['stroop', 'n-back', 'mat', 'hanoi']):
        return 'cognitive_load'
    if any(x in name for x in ['vr-plank']):
        return 'high_stress'
    if any(x in name for x in ['manual', 'cobot']):
        return 'industrial_task'
    if any(x in name for x in ['rest', 'meditation']):
        return 'low_load'
    return 'other'


labels_df = modality_tables.get('labels')

if labels_df is not None:
    print('✓ Using labels.csv as primary label source')
    # Keep as-is for now; schema can vary by release.
    # We expose labels table but avoid forcing a brittle join.
else:
    print('⚠️ labels.csv not found')
    if CONFIG.create_pseudo_labels_if_missing and CONFIG.pseudo_label_strategy == 'task_rule':
        pseudo = physio_df[['subject_id', 'task_name', 'task_file']].drop_duplicates().copy()
        pseudo['pseudo_label'] = pseudo['task_name'].map(create_task_rule_label)
        labels_df = pseudo
        print('✓ Pseudo-labels created from task names')
    elif CONFIG.require_labels:
        raise FileNotFoundError('labels.csv missing and require_labels=True')

display(labels_df.head(10) if labels_df is not None else pd.DataFrame({'labels': ['missing']}))

⚠️ labels.csv not found
✓ Pseudo-labels created from task names


,subject_id,task_name,task_file,pseudo_label
0,10000,cobot-task-1,cobot-task-1.csv,industrial_task
12,10000,cobot-task-2,cobot-task-2.csv,industrial_task
32,10000,cobot-task-3,cobot-task-3.csv,industrial_task
40,10000,cobot-task-4,cobot-task-4.csv,industrial_task
50,10000,cobot-task-5,cobot-task-5.csv,industrial_task
56,10000,hanoi_0,hanoi_0.csv,cognitive_load
71,10000,manual-task-1,manual-task-1.csv,industrial_task
85,10000,manual-task-2,manual-task-2.csv,industrial_task
94,10000,manual-task-3,manual-task-3.csv,industrial_task
102,10000,manual-task-4,manual-task-4.csv,industrial_task


In [6]:
# SECTION 6: Key Harmonization Helpers


def infer_join_keys(left: pd.DataFrame, right: pd.DataFrame, preferred: List[str]) -> List[str]:
    keys = [k for k in preferred if k in left.columns and k in right.columns]
    return keys


def prefix_feature_columns(df: pd.DataFrame, prefix: str, protected_cols: List[str]) -> pd.DataFrame:
    rename_map = {
        c: f'{prefix}__{c}'
        for c in df.columns
        if c not in protected_cols
    }
    return df.rename(columns=rename_map)


def normalize_key_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Map common schema variants to canonical key names.
    rename_candidates = {
        'subject': 'subject_id',
        'subj': 'subject_id',
        'participant': 'subject_id',
        'participant_id': 'subject_id',
        'task': 'task_name',
        'taskid': 'task_name',
        'task_id': 'task_name',
        'filename': 'task_file',
        'file_name': 'task_file',
        'file': 'task_file',
        'window': 'window_idx',
        'window_id': 'window_idx',
        'start': 'start_idx',
        'end': 'end_idx',
    }

    lower_map = {c.lower(): c for c in out.columns}
    for src_l, dst in rename_candidates.items():
        if dst in out.columns:
            continue
        if src_l in lower_map:
            out = out.rename(columns={lower_map[src_l]: dst})

    return out


preferred_keys = ['subject_id', 'task_file', 'task_name', 'window_idx', 'start_idx', 'end_idx']
physio_keys = ['subject_id', 'task_file', 'task_name', 'window_idx', 'start_idx', 'end_idx']

print('✓ Harmonization helpers ready')

✓ Harmonization helpers ready


In [7]:
# SECTION 7: Build Fusion Dataset (Left Join on Physiological Windows)

fusion_df = normalize_key_columns(physio_df.copy())
joined_modalities = []

# Attach labels (or pseudo-labels) at task-level where possible
if labels_df is not None and not labels_df.empty:
    label_payload = normalize_key_columns(labels_df.copy())
    label_keys = infer_join_keys(fusion_df, label_payload, ['subject_id', 'task_file', 'task_name'])
    if label_keys:
        # Avoid accidental massive duplicate columns
        keep_cols = list(dict.fromkeys(label_keys + [c for c in label_payload.columns if c not in label_keys]))
        label_payload = label_payload[keep_cols]
        fusion_df = fusion_df.merge(label_payload, on=label_keys, how='left', suffixes=('', '__label'))
        joined_modalities.append('labels')
    else:
        print('⚠️ labels table found but no compatible join keys after normalization')

# Join optional modality features if keys are compatible
optional_modalities = ['eeg_features_5s', 'speech_features', 'aus_data', 'nlp_embeddings']
for mod in optional_modalities:
    table = modality_tables.get(mod)
    if table is None or table.empty:
        continue

    table = normalize_key_columns(table.copy())
    join_keys = infer_join_keys(fusion_df, table, preferred_keys)
    if not join_keys:
        print(f'⚠️ Skipping {mod}: no common join keys (columns: {list(table.columns)[:12]}...)')
        continue

    protected = join_keys
    prefixed = prefix_feature_columns(table.copy(), mod, protected)
    # Drop exact duplicate rows on join keys to avoid row explosion
    prefixed = prefixed.drop_duplicates(subset=join_keys)

    before = len(fusion_df)
    fusion_df = fusion_df.merge(prefixed, on=join_keys, how='left')
    after = len(fusion_df)

    print(f'✓ Joined {mod} on keys {join_keys} | rows {before} -> {after}')
    joined_modalities.append(mod)

print('\n✓ Fusion table built')
print(f'  Shape: {fusion_df.shape}')
print(f'  Joined modalities: {joined_modalities}')
display(fusion_df.head(3))

✓ Joined eeg_features_5s on keys ['subject_id', 'task_file', 'task_name', 'window_idx', 'start_idx', 'end_idx'] | rows 5640 -> 5640

✓ Fusion table built
  Shape: (5640, 251)
  Joined modalities: ['labels', 'eeg_features_5s']


,subject_id,task_name,task_file,split,ID__mean,ID__std,ID__min,ID__max,ID__median,ID__p25,ID__p75,ID__energy,Repetition__mean,Repetition__std,Repetition__min,Repetition__max,Repetition__median,Repetition__p25,Repetition__p75,Repetition__energy,ECG__mean,ECG__std,ECG__min,ECG__max,ECG__median,ECG__p25,ECG__p75,ECG__energy,EDA__mean,EDA__std,EDA__min,EDA__max,EDA__median,EDA__p25,EDA__p75,EDA__energy,EMG__mean,EMG__std,EMG__min,EMG__max,EMG__median,EMG__p25,EMG__p75,EMG__energy,RESP__mean,RESP__std,RESP__min,RESP__max,RESP__median,RESP__p25,RESP__p75,RESP__energy,EEG_channel_0__mean,EEG_channel_0__std,EEG_channel_0__min,EEG_channel_0__max,EEG_channel_0__median,EEG_channel_0__p25,EEG_channel_0__p75,EEG_channel_0__energy,EEG_channel_1__mean,EEG_channel_1__std,EEG_channel_1__min,EEG_channel_1__max,EEG_channel_1__median,EEG_channel_1__p25,EEG_channel_1__p75,EEG_channel_1__energy,EEG_channel_2__mean,EEG_channel_2__std,EEG_channel_2__min,EEG_channel_2__max,EEG_channel_2__median,EEG_channel_2__p25,EEG_channel_2__p75,EEG_channel_2__energy,EEG_channel_3__mean,EEG_channel_3__std,EEG_channel_3__min,EEG_channel_3__max,EEG_channel_3__median,EEG_channel_3__p25,EEG_channel_3__p75,EEG_channel_3__energy,EEG_channel_4__mean,EEG_channel_4__std,EEG_channel_4__min,EEG_channel_4__max,EEG_channel_4__median,EEG_channel_4__p25,EEG_channel_4__p75,EEG_channel_4__energy,EEG_channel_5__mean,EEG_channel_5__std,EEG_channel_5__min,EEG_channel_5__max,EEG_channel_5__median,EEG_channel_5__p25,EEG_channel_5__p75,EEG_channel_5__energy,...,n_samples,pseudo_label,eeg_features_5s__split,eeg_features_5s__n_samples,eeg_features_5s__EEG_channel_0__mean,eeg_features_5s__EEG_channel_0__std,eeg_features_5s__EEG_channel_0__min,eeg_features_5s__EEG_channel_0__max,eeg_features_5s__EEG_channel_0__median,eeg_features_5s__EEG_channel_0__p25,eeg_features_5s__EEG_channel_0__p75,eeg_features_5s__EEG_channel_0__energy,eeg_features_5s__EEG_channel_1__mean,eeg_features_5s__EEG_channel_1__std,eeg_features_5s__EEG_channel_1__min,eeg_features_5s__EEG_channel_1__max,eeg_features_5s__EEG_channel_1__median,eeg_features_5s__EEG_channel_1__p25,eeg_features_5s__EEG_channel_1__p75,eeg_features_5s__EEG_channel_1__energy,eeg_features_5s__EEG_channel_2__mean,eeg_features_5s__EEG_channel_2__std,eeg_features_5s__EEG_channel_2__min,eeg_features_5s__EEG_channel_2__max,eeg_features_5s__EEG_channel_2__median,eeg_features_5s__EEG_channel_2__p25,eeg_features_5s__EEG_channel_2__p75,eeg_features_5s__EEG_channel_2__energy,eeg_features_5s__EEG_channel_3__mean,eeg_features_5s__EEG_channel_3__std,eeg_features_5s__EEG_channel_3__min,eeg_features_5s__EEG_channel_3__max,eeg_features_5s__EEG_channel_3__median,eeg_features_5s__EEG_channel_3__p25,eeg_features_5s__EEG_channel_3__p75,eeg_features_5s__EEG_channel_3__energy,eeg_features_5s__EEG_channel_4__mean,eeg_features_5s__EEG_channel_4__std,eeg_features_5s__EEG_channel_4__min,eeg_features_5s__EEG_channel_4__max,eeg_features_5s__EEG_channel_4__median,eeg_features_5s__EEG_channel_4__p25,eeg_features_5s__EEG_channel_4__p75,eeg_features_5s__EEG_channel_4__energy,eeg_features_5s__EEG_channel_5__mean,eeg_features_5s__EEG_channel_5__std,eeg_features_5s__EEG_channel_5__min,eeg_features_5s__EEG_channel_5__max,eeg_features_5s__EEG_channel_5__median,eeg_features_5s__EEG_channel_5__p25,eeg_features_5s__EEG_channel_5__p75,eeg_features_5s__EEG_channel_5__energy,eeg_features_5s__EEG_channel_6__mean,eeg_features_5s__EEG_channel_6__std,eeg_features_5s__EEG_channel_6__min,eeg_features_5s__EEG_channel_6__max,eeg_features_5s__EEG_channel_6__median,eeg_features_5s__EEG_channel_6__p25,eeg_features_5s__EEG_channel_6__p75,eeg_features_5s__EEG_channel_6__energy,eeg_features_5s__EEG_channel_7__mean,eeg_features_5s__EEG_channel_7__std,eeg_features_5s__EEG_channel_7__min,eeg_features_5s__EEG_channel_7__max,eeg_features_5s__EEG_channel_7__median,eeg_features_5s__EEG_channel_7__p25,eeg_features_5s__EEG_channel_7__p75,eeg_features_5s__EEG_channel_7__energy,eeg_features_5s__EEG_channel_8__mean,eeg_

In [8]:
# SECTION 8: Modality Presence Flags and Completeness Diagnostics


def add_presence_flag(df: pd.DataFrame, prefix: str, flag_name: str) -> pd.Series:
    cols = [c for c in df.columns if c.startswith(prefix + '__')]
    if not cols:
        return pd.Series(0, index=df.index)
    return (~df[cols].isna().all(axis=1)).astype(int)


fusion_df['has_physio'] = 1
fusion_df['has_eeg'] = add_presence_flag(fusion_df, 'eeg_features_5s', 'has_eeg')
fusion_df['has_speech'] = add_presence_flag(fusion_df, 'speech_features', 'has_speech')
fusion_df['has_au'] = add_presence_flag(fusion_df, 'aus_data', 'has_au')
fusion_df['has_nlp'] = add_presence_flag(fusion_df, 'nlp_embeddings', 'has_nlp')

fusion_df['n_modalities_present'] = fusion_df[['has_physio', 'has_eeg', 'has_speech', 'has_au', 'has_nlp']].sum(axis=1)

before = len(fusion_df)
fusion_df = fusion_df[fusion_df['n_modalities_present'] >= CONFIG.min_modalities_required].reset_index(drop=True)
after = len(fusion_df)

print('✓ Presence flags computed')
print(f'  Rows before min-modality filter: {before}')
print(f'  Rows after  min-modality filter: {after}')

presence_summary = fusion_df[['has_physio', 'has_eeg', 'has_speech', 'has_au', 'has_nlp', 'n_modalities_present']].describe().T
print('\nPresence summary:')
display(presence_summary)

max_modalities = int(fusion_df['n_modalities_present'].max()) if len(fusion_df) else 0
if CONFIG.expect_multimodal and max_modalities <= 1:
    msg = (
        'No non-physio modalities were joined. This run is physio-only. '\
        'Check modality file paths/names or generate modality feature CSVs first.'
    )
    if CONFIG.strict_multimodal_check:
        raise ValueError(msg)
    print(f'⚠️ {msg}')

✓ Presence flags computed
  Rows before min-modality filter: 5640
  Rows after  min-modality filter: 5640

Presence summary:


,count,mean,std,min,25%,50%,75%,max
has_physio,5640.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0
has_eeg,5640.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0
has_speech,5640.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
has_au,5640.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
has_nlp,5640.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
n_modalities_present,5640.0,2.0,0.0,2.0,2.0,2.0,2.0,2.0


In [9]:
# SECTION 9: Split Blueprint for Training (LOSO-Compatible)

subject_counts = fusion_df.groupby('subject_id').size().sort_values(ascending=False)

split_blueprint = pd.DataFrame({
    'subject_id': subject_counts.index,
    'n_rows': subject_counts.values,
})

split_blueprint['test_ratio_if_loso'] = split_blueprint['n_rows'] / split_blueprint['n_rows'].sum()

print('✓ Split blueprint generated')
print(f'  Subjects: {len(split_blueprint)}')
print(f"  Test ratio range (LOSO): {split_blueprint['test_ratio_if_loso'].min():.4f} - {split_blueprint['test_ratio_if_loso'].max():.4f}")

display(split_blueprint.head(10))

✓ Split blueprint generated
  Subjects: 52
  Test ratio range (LOSO): 0.0032 - 0.0289


,subject_id,n_rows,test_ratio_if_loso
0,1032,163,0.028901
1,10000,163,0.028901
2,1038,160,0.028369
3,1065,158,0.028014
4,1035,157,0.027837
5,1051,155,0.027482
6,1005,155,0.027482
7,1008,153,0.027128
8,1058,152,0.026950
9,10001,152,0.026950


In [10]:
# SECTION 10: Export Fusion Dataset and Manifest

export_path = OUT_DIR / 'fusion_dataset.csv'
split_path = OUT_DIR / 'fusion_split_blueprint.csv'
manifest_path = OUT_DIR / 'fusion_manifest.json'

fusion_df.to_csv(export_path, index=False)
split_blueprint.to_csv(split_path, index=False)

manifest = {
    'config': asdict(CONFIG),
    'dataset_root': str(DATASET_ROOT),
    'preproc_root': str(PREPROC_ROOT),
    'features_root_exists': FEATURES_ROOT.exists(),
    'rows': int(len(fusion_df)),
    'subjects': int(fusion_df.subject_id.nunique()),
    'columns': int(fusion_df.shape[1]),
    'joined_modalities': joined_modalities,
    'presence_rates': {
        'has_physio': float(fusion_df['has_physio'].mean()),
        'has_eeg': float(fusion_df['has_eeg'].mean()),
        'has_speech': float(fusion_df['has_speech'].mean()),
        'has_au': float(fusion_df['has_au'].mean()),
        'has_nlp': float(fusion_df['has_nlp'].mean()),
    },
    'files': {
        'fusion_dataset': str(export_path),
        'split_blueprint': str(split_path),
    },
}

with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print('✓ Fusion dataset exported')
print(f'  - {export_path}')
print(f'  - {split_path}')
print(f'  - {manifest_path}')
print('\nManifest preview:')
print(json.dumps(manifest, indent=2))

✓ Fusion dataset exported
  - /content/drive/MyDrive/research_outputs/fusion/v1_fusion/fusion_dataset.csv
  - /content/drive/MyDrive/research_outputs/fusion/v1_fusion/fusion_split_blueprint.csv
  - /content/drive/MyDrive/research_outputs/fusion/v1_fusion/fusion_manifest.json

Manifest preview:
{
  "config": {
    "use_normalized_physio": true,
    "require_labels": false,
    "create_pseudo_labels_if_missing": true,
    "pseudo_label_strategy": "task_rule",
    "min_modalities_required": 1,
    "expect_multimodal": true,
    "strict_multimodal_check": false,
    "output_version": "v1_fusion"
  },
  "dataset_root": "/content/drive/MyDrive/MultiPhysio-HRC",
  "preproc_root": "/content/drive/MyDrive/research_outputs/preprocessing/v1_windowed",
  "features_root_exists": true,
  "rows": 5640,
  "subjects": 52,
  "columns": 257,
  "joined_modalities": [
    "labels",
    "eeg_features_5s"
  ],
  "presence_rates": {
    "has_physio": 1.0,
    "has_eeg": 1.0,
    "has_speech": 0.0,
    "has_au

## Next Steps

1. Train baseline models on the exported fusion dataset with LOSO.
2. Compare unimodal vs early fusion vs late fusion.
3. If true questionnaire labels become available, replace pseudo-labels and rerun this notebook.
4. Add temporal sequence builders (per-task ordered windows) for transformer/LSTM fusion models.

In [11]:
# Compact run summary for quick evaluation
import pandas as pd

summary = {}

if 'physio_df' in globals() and isinstance(physio_df, pd.DataFrame):
    summary['physio_rows'] = int(len(physio_df))
    summary['physio_cols'] = int(len(physio_df.columns))
else:
    summary['physio_rows'] = None
    summary['physio_cols'] = None

if 'fusion_df' in globals() and isinstance(fusion_df, pd.DataFrame):
    summary['fusion_rows'] = int(len(fusion_df))
    summary['fusion_cols'] = int(len(fusion_df.columns))

    core_keys = [k for k in ['subject', 'task', 'window_idx'] if k in fusion_df.columns]
    summary['core_keys_present'] = core_keys

    if 'subject' in fusion_df.columns:
        summary['fusion_subjects'] = int(fusion_df['subject'].nunique())
    if 'task' in fusion_df.columns:
        summary['fusion_tasks'] = int(fusion_df['task'].nunique())
else:
    summary['fusion_rows'] = None
    summary['fusion_cols'] = None
    summary['core_keys_present'] = []

if 'presence_summary' in globals() and isinstance(presence_summary, pd.DataFrame) and not presence_summary.empty:
    # Try to infer expected columns robustly
    cols = {c.lower(): c for c in presence_summary.columns}
    mod_col = cols.get('modality', None)
    if mod_col is None:
        for c in presence_summary.columns:
            if 'mod' in c.lower():
                mod_col = c
                break

    pct_col = None
    for c in presence_summary.columns:
        if 'pct' in c.lower() or 'rate' in c.lower() or 'coverage' in c.lower():
            pct_col = c
            break

    summary['presence_modalities'] = int(len(presence_summary))
    if mod_col and pct_col:
        best = presence_summary.sort_values(pct_col, ascending=False).head(3)
        worst = presence_summary.sort_values(pct_col, ascending=True).head(3)
        summary['best_modalities'] = best[[mod_col, pct_col]].to_dict(orient='records')
        summary['worst_modalities'] = worst[[mod_col, pct_col]].to_dict(orient='records')
else:
    summary['presence_modalities'] = None

if 'split_blueprint' in globals() and isinstance(split_blueprint, pd.DataFrame) and not split_blueprint.empty:
    summary['split_rows'] = int(len(split_blueprint))
    if {'split', 'n_rows'}.issubset(set(split_blueprint.columns)):
        summary['split_counts'] = dict(zip(split_blueprint['split'].astype(str), split_blueprint['n_rows'].astype(int)))
else:
    summary['split_rows'] = None

if 'manifest_path' in globals():
    summary['manifest_path'] = str(manifest_path)
if 'export_path' in globals():
    summary['fusion_export_path'] = str(export_path)
if 'split_path' in globals():
    summary['split_export_path'] = str(split_path)

print('=== FUSION NOTEBOOK COMPACT SUMMARY ===')
for k, v in summary.items():
    print(f'{k}: {v}')


=== FUSION NOTEBOOK COMPACT SUMMARY ===
physio_rows: 5640
physio_cols: 152
fusion_rows: 5640
fusion_cols: 257
core_keys_present: ['window_idx']
presence_modalities: 6
split_rows: 52
manifest_path: /content/drive/MyDrive/research_outputs/fusion/v1_fusion/fusion_manifest.json
fusion_export_path: /content/drive/MyDrive/research_outputs/fusion/v1_fusion/fusion_dataset.csv
split_export_path: /content/drive/MyDrive/research_outputs/fusion/v1_fusion/fusion_split_blueprint.csv


In [ ]:
# Additional integrity diagnostics
import pandas as pd

print('=== FUSION INTEGRITY CHECK ===')

if 'fusion_df' in globals() and isinstance(fusion_df, pd.DataFrame):
    print('fusion columns sample:', fusion_df.columns[:20].tolist())
    print('has subject:', 'subject' in fusion_df.columns)
    print('has task:', 'task' in fusion_df.columns)
    print('has window_idx:', 'window_idx' in fusion_df.columns)

    if 'label' in fusion_df.columns:
        vc = fusion_df['label'].value_counts(dropna=False)
        print('label distribution (top 10):')
        print(vc.head(10))

if 'split_blueprint' in globals() and isinstance(split_blueprint, pd.DataFrame):
    print('\nsplit_blueprint columns:', split_blueprint.columns.tolist())
    print('split_blueprint head:')
    print(split_blueprint.head())

if 'presence_summary' in globals() and isinstance(presence_summary, pd.DataFrame):
    print('\npresence_summary columns:', presence_summary.columns.tolist())
    print(presence_summary)


=== FUSION INTEGRITY CHECK ===
fusion columns sample: ['subject_id', 'task_name', 'task_file', 'split', 'ID__mean', 'ID__std', 'ID__min', 'ID__max', 'ID__median', 'ID__p25', 'ID__p75', 'ID__energy', 'Repetition__mean', 'Repetition__std', 'Repetition__min', 'Repetition__max', 'Repetition__median', 'Repetition__p25', 'Repetition__p75', 'Repetition__energy']
has subject: False
has task: False
has window_idx: True

split_blueprint columns: ['subject_id', 'n_rows', 'test_ratio_if_loso']
split_blueprint head:
   subject_id  n_rows  test_ratio_if_loso
0        1032     163            0.028901
1       10000     163            0.028901
2        1038     160            0.028369
3        1065     158            0.028014
4        1035     157            0.027837

presence_summary columns: ['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']
                       count  mean  std  min  25%  50%  75%  max
has_physio            5640.0   1.0  0.0  1.0  1.0  1.0  1.0  1.0
has_eeg               

In [13]:
# Why presence flags are zero: compact join diagnostics
import pandas as pd

print('=== JOIN DIAGNOSTICS ===')

base = physio_df.copy()
mods = ['eeg_features_5s', 'speech_features', 'aus_data', 'nlp_embeddings']

for mod in mods:
    table = modality_tables.get(mod)
    print(f'\n[{mod}]')
    if table is None:
        print('status: file missing')
        continue
    if table.empty:
        print('status: table exists but empty')
        continue

    print(f'table shape: {table.shape}')
    print('table key candidates:', [c for c in ['subject_id','task_name','task_file','window_idx','start_idx','end_idx','split'] if c in table.columns])

    join_keys = infer_join_keys(base, table, preferred_keys)
    print('inferred join keys:', join_keys)

    if not join_keys:
        print('result: cannot join (no common keys)')
        continue

    left_keys = base[join_keys].drop_duplicates()
    right_keys = table[join_keys].drop_duplicates()
    overlap = left_keys.merge(right_keys, on=join_keys, how='inner')

    print('unique base keys:', len(left_keys))
    print('unique mod keys :', len(right_keys))
    print('key overlap rows:', len(overlap))

    if len(left_keys) > 0:
        print(f'overlap ratio vs base keys: {len(overlap)/len(left_keys):.4%}')

# Also report non-null rates for already-joined columns in fusion_df
print('\n=== NON-NULL RATE OF JOINED FEATURE GROUPS IN fusion_df ===')
for prefix in mods:
    cols = [c for c in fusion_df.columns if c.startswith(prefix + '__')]
    if not cols:
        print(f'{prefix}: no prefixed columns present in fusion_df')
        continue
    rate = (~fusion_df[cols].isna().all(axis=1)).mean()
    print(f'{prefix}: rows with any non-null feature = {rate:.4%} ({len(cols)} cols)')


=== JOIN DIAGNOSTICS ===

[eeg_features_5s]
table shape: (5640, 104)
table key candidates: ['subject_id', 'task_name', 'task_file', 'window_idx', 'start_idx', 'end_idx', 'split']
inferred join keys: ['subject_id', 'task_file', 'task_name', 'window_idx', 'start_idx', 'end_idx']
unique base keys: 5640
unique mod keys : 5640
key overlap rows: 5640
overlap ratio vs base keys: 100.0000%

[speech_features]
status: file missing

[aus_data]
status: file missing

[nlp_embeddings]
status: file missing

=== NON-NULL RATE OF JOINED FEATURE GROUPS IN fusion_df ===
eeg_features_5s: rows with any non-null feature = 100.0000% (98 cols)
speech_features: no prefixed columns present in fusion_df
aus_data: no prefixed columns present in fusion_df
nlp_embeddings: no prefixed columns present in fusion_df


In [14]:
# Dataset tree probe for modality evidence
from pathlib import Path

print('=== DATASET TREE PROBE ===')
print('DATASET_ROOT:', DATASET_ROOT)

if not DATASET_ROOT.exists():
    print('DATASET_ROOT does not exist in this runtime.')
else:
    interesting = []
    for p in DATASET_ROOT.rglob('*'):
        if len(interesting) >= 200:
            break
        name = p.name.lower()
        if any(k in name for k in ['eeg', 'speech', 'audio', 'au', 'action', 'nlp', 'text', 'embed', 'label', 'feature']):
            interesting.append(p)

    print('Interesting paths found (up to 200):')
    for p in interesting:
        print(' -', p)

    csvs = [p for p in DATASET_ROOT.rglob('*.csv')][:200]
    print(f'\nCSV files sample ({len(csvs)} shown, max 200):')
    for p in csvs:
        print(' -', p)


=== DATASET TREE PROBE ===
DATASET_ROOT: /content/drive/MyDrive/MultiPhysio-HRC
Interesting paths found (up to 200):
 - /content/drive/MyDrive/MultiPhysio-HRC/features
 - /content/drive/MyDrive/MultiPhysio-HRC/features/eeg_features_5s.csv
 - /content/drive/MyDrive/MultiPhysio-HRC/features/ecg_features.csv
 - /content/drive/MyDrive/MultiPhysio-HRC/features/eda_features.csv
 - /content/drive/MyDrive/MultiPhysio-HRC/features/emg_features.csv
 - /content/drive/MyDrive/MultiPhysio-HRC/features/resp_features.csv

CSV files sample (200 shown, max 200):
 - /content/drive/MyDrive/MultiPhysio-HRC/features/eeg_features_5s.csv
 - /content/drive/MyDrive/MultiPhysio-HRC/features/ecg_features.csv
 - /content/drive/MyDrive/MultiPhysio-HRC/features/eda_features.csv
 - /content/drive/MyDrive/MultiPhysio-HRC/features/emg_features.csv
 - /content/drive/MyDrive/MultiPhysio-HRC/features/resp_features.csv
 - /content/drive/MyDrive/MultiPhysio-HRC/features/published_modality_tables.csv
 - /content/drive/MyDri

In [15]:
# Compact modality evidence summary
from collections import Counter

print('=== COMPACT MODALITY EVIDENCE ===')

if not DATASET_ROOT.exists():
    print('DATASET_ROOT missing')
else:
    modality_hits = []
    csv_hits = []
    for p in DATASET_ROOT.rglob('*'):
        if p.is_file():
            lp = str(p).lower()
            if lp.endswith('.csv'):
                csv_hits.append(p)
            if any(k in lp for k in ['eeg', 'speech', 'audio', 'au', 'action_unit', 'nlp', 'embed', 'label']):
                modality_hits.append(p)

    print('total csv files:', len(csv_hits))
    print('modality-like files:', len(modality_hits))

    # Show only first 20 modality-like hits for readability
    print('sample modality-like files (max 20):')
    for p in modality_hits[:20]:
        print(' -', p)

    # Count parent folder names for quick location hint
    parents = Counter([p.parent.name for p in modality_hits])
    print('top parent folders:', parents.most_common(10))


=== COMPACT MODALITY EVIDENCE ===
total csv files: 1844
modality-like files: 1
sample modality-like files (max 20):
 - /content/drive/MyDrive/MultiPhysio-HRC/features/eeg_features_5s.csv
top parent folders: [('features', 1)]


In [11]:
# SECTION 12: Deep Exploration of Built Fusion Dataset

print('=== DEEP FUSION DATASET ANALYSIS ===')

if 'fusion_df' not in globals() or fusion_df.empty:
    raise ValueError('fusion_df is empty. Run build sections first.')

# 1) Shape and key integrity
print('\n[1] SHAPE + KEY INTEGRITY')
print('rows:', len(fusion_df))
print('columns:', len(fusion_df.columns))

key_candidates = ['subject_id', 'task_name', 'task_file', 'window_idx', 'start_idx', 'end_idx']
keys_present = [k for k in key_candidates if k in fusion_df.columns]
print('keys_present:', keys_present)

if keys_present:
    dup_keys = int(fusion_df.duplicated(subset=keys_present).sum())
    print('duplicate_rows_on_keys:', dup_keys)

# 2) Missingness overview
print('\n[2] MISSINGNESS OVERVIEW')
missing = fusion_df.isna().mean().sort_values(ascending=False)
print('columns_with_any_missing:', int((missing > 0).sum()))
print('top_15_missing_columns:')
print(missing.head(15).to_string())

# 3) Modality coverage
print('\n[3] MODALITY COVERAGE')
presence_cols = [c for c in ['has_physio', 'has_eeg', 'has_speech', 'has_au', 'has_nlp', 'n_modalities_present'] if c in fusion_df.columns]
if presence_cols:
    print(fusion_df[presence_cols].describe().T.to_string())

# 4) Subject/task distribution
print('\n[4] SUBJECT/TASK DISTRIBUTION')
if 'subject_id' in fusion_df.columns:
    subj_counts = fusion_df['subject_id'].value_counts()
    print('subjects:', int(subj_counts.shape[0]))
    print('rows_per_subject min/mean/max:', int(subj_counts.min()), float(subj_counts.mean()), int(subj_counts.max()))

if 'task_name' in fusion_df.columns:
    task_counts = fusion_df['task_name'].value_counts()
    print('tasks:', int(task_counts.shape[0]))
    print('rows_per_task min/mean/max:', int(task_counts.min()), float(task_counts.mean()), int(task_counts.max()))
    print('top_10_tasks:')
    print(task_counts.head(10).to_string())

# 5) Label distribution
print('\n[5] LABEL DISTRIBUTION')
label_col = None
for c in ['label', 'pseudo_label']:
    if c in fusion_df.columns:
        label_col = c
        break

if label_col is not None:
    y = fusion_df[label_col].fillna('NA').astype(str)
    dist = y.value_counts()
    print('label_column_used:', label_col)
    print(dist.to_string())
    print('label_entropy_bits:', float(-(dist / dist.sum() * np.log2(dist / dist.sum())).sum()))
else:
    print('No label column found.')

# 6) Feature space summary
print('\n[6] FEATURE SPACE SUMMARY')
meta_like = set(['subject_id', 'task_name', 'task_file', 'split', 'window_idx', 'start_idx', 'end_idx', 'n_samples',
                 'has_physio', 'has_eeg', 'has_speech', 'has_au', 'has_nlp', 'n_modalities_present', 'pseudo_label', 'label'])
feature_cols = [c for c in fusion_df.columns if c not in meta_like]
print('feature_columns_count:', len(feature_cols))

prefix_counts = {}
for c in feature_cols:
    pref = c.split('__')[0]
    prefix_counts[pref] = prefix_counts.get(pref, 0) + 1

top_prefix = sorted(prefix_counts.items(), key=lambda x: x[1], reverse=True)[:20]
print('top_feature_prefix_counts:')
for p, n in top_prefix:
    print(f'  {p}: {n}')

# 7) Save analysis snapshot
analysis_out = OUT_DIR / 'fusion_analysis_snapshot.json'
snapshot = {
    'rows': int(len(fusion_df)),
    'columns': int(len(fusion_df.columns)),
    'keys_present': keys_present,
    'duplicate_rows_on_keys': int(fusion_df.duplicated(subset=keys_present).sum()) if keys_present else None,
    'columns_with_any_missing': int((fusion_df.isna().sum() > 0).sum()),
    'subjects': int(fusion_df['subject_id'].nunique()) if 'subject_id' in fusion_df.columns else None,
    'tasks': int(fusion_df['task_name'].nunique()) if 'task_name' in fusion_df.columns else None,
    'presence_rates': {
        c: float(fusion_df[c].mean()) for c in ['has_physio', 'has_eeg', 'has_speech', 'has_au', 'has_nlp'] if c in fusion_df.columns
    },
    'feature_columns_count': int(len(feature_cols)),
}

with open(analysis_out, 'w') as f:
    json.dump(snapshot, f, indent=2)

print('\nSaved analysis snapshot:', analysis_out)


=== DEEP FUSION DATASET ANALYSIS ===

[1] SHAPE + KEY INTEGRITY
rows: 5640
columns: 257
keys_present: ['subject_id', 'task_name', 'task_file', 'window_idx', 'start_idx', 'end_idx']
duplicate_rows_on_keys: 0

[2] MISSINGNESS OVERVIEW
columns_with_any_missing: 8
top_15_missing_columns:
EDA__energy                               0.00727
EDA__p75                                  0.00727
EDA__p25                                  0.00727
EDA__median                               0.00727
EDA__max                                  0.00727
EDA__min                                  0.00727
EDA__std                                  0.00727
EDA__mean                                 0.00727
eeg_features_5s__EEG_channel_0__p25       0.00000
eeg_features_5s__EEG_channel_2__mean      0.00000
eeg_features_5s__EEG_channel_1__energy    0.00000
eeg_features_5s__EEG_channel_1__p75       0.00000
eeg_features_5s__EEG_channel_1__p25       0.00000
eeg_features_5s__EEG_channel_1__median    0.00000
eeg_features_5s